# Evaluation of 3 Retrieval Approaches
1. Vector search using cosine similarity

2. Hybrid search: BM25 + vector search using cosine similarity

3. Hybrid search + reranking: BM25 + vector search using cosine similarity + reranking

In [1]:
import json
import pandas as pd
from IPython.display import display

from support_ops.helpers.config import ROOT, load_config
from support_ops.policy.rag.embeddings import read_embeddings
from support_ops.policy.rag.retriever import APPROACHES, RagSettings, shared_retriever

settings = RagSettings.from_config(load_config()["policy"]["rag"])
manifest, documents = read_embeddings()
cases = json.loads((ROOT / "data/cases/rag/questions.json").read_text())
ids = {doc["_id"] for doc in documents}
assert cases and len({case["id"] for case in cases}) == len(cases)
for case in cases:
    assert case["expected_chunk_ids"] and not set(case["expected_chunk_ids"]) - ids, case["id"]
    assert not set(case.get("expected_reference_chunk_ids", [])) - ids, case["id"]

# Evaluation metrics
- Recall@4: Fraction of expected chunks found in the top 4.
- MRR@4: Average of 1 / rank of the first correct chunk; 0 if none is in the top 4.
- nDCG@4: Rewards relevant chunks appearing nearer the top.

Expected chunks have relevance 1; other chunks have relevance 0.
These scores exclude extra reference fetches.

Reference checks separately verify that labelled linked chunks are in the complete response.
A linked chunk already in the top 4 counts as present; it does not need fetching again.
A failed reference check can mean the initial search missed its source clause.
The questions are one comparison set, with no separate held-out test set.

In [2]:
from ranx import evaluate

# Compare expected chunk IDs with the returned order; ranx calculates the metrics.
def retrieval_metrics(expected_ids, retrieved_ids, k):
    expected = {"query": {key: 1 for key in expected_ids}}
    # ranx needs scores as input; these numbers preserve the order already returned by retrieval.
    retrieved = {"query": {key: float(k - rank) for rank, key in enumerate(retrieved_ids[:k])}}
    scores = evaluate(expected, retrieved, [f"recall@{k}", f"mrr@{k}", f"ndcg@{k}"])
    return {name.replace(f"@{k}", "_at_k"): float(value) for name, value in scores.items()}

In [3]:
# Check linked chunks separately from the top-K ranking metrics.
def check_references(expected_ids, hits):
    returned = {hit["id"] for hit in hits}
    missing = sorted(set(expected_ids) - returned)
    return {
        "expected_reference_chunk_ids": expected_ids,
        "missing_reference_chunk_ids": missing,
        "references_returned": not missing if expected_ids else None,
    }

In [4]:
# Only evaluation selects different approaches; agent tools use retrieve(question).
retrievers = {
    name: shared_retriever(RagSettings.from_config(
        load_config(f"configs/rag/{name}.yaml")["policy"]["rag"]
    ))
    for name in APPROACHES
}

# Run every question through every approach with the same settings.
rows = []
for case in cases:
    for name, retriever in retrievers.items():
        hits = retriever.search(case["question"])
        retrieved = [hit["id"] for hit in hits if hit["via"] == "search"]
        references = [hit["id"] for hit in hits if hit["via"] != "search"]
        rows.append({
            "case_id": case["id"], "question": case["question"], "approach": name,
            "expected_chunk_ids": case["expected_chunk_ids"],
            "retrieved_chunk_ids": retrieved, "reference_chunk_ids": references,
            **retrieval_metrics(case["expected_chunk_ids"], retrieved, retriever.settings.k),
            **check_references(case.get("expected_reference_chunk_ids", []), hits),
        })

# Show the average scores for each approach.
results = pd.DataFrame(rows)
metrics = ["recall_at_k", "mrr_at_k", "ndcg_at_k"]
summary = results.groupby("approach", sort=False)[metrics].mean().round(3)
summary.insert(0, "questions", len(cases))

display(summary)

/Users/hanpeiying/Desktop/Gen AI with LLM/GenAIProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1031 03:05:20.933000 98227 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7327.29it/s]


,questions,recall_at_k,mrr_at_k,ndcg_at_k
approach,,,,
vector,42,0.917,0.823,0.840
hybrid,42,0.917,0.782,0.808
hybrid_rerank,42,0.988,0.909,0.926


Sanity check if Reference files ar retrieved where they exist

In [7]:
# Show the labelled reference checks separately from the ranking summary.
reference_checks = results.loc[results["references_returned"].notna(), [
    "case_id", "approach", "expected_reference_chunk_ids",
    "missing_reference_chunk_ids", "references_returned",
]]
with pd.option_context("display.max_colwidth", None):
    display(reference_checks.reset_index(drop=True))

,case_id,approach,expected_reference_chunk_ids,missing_reference_chunk_ids,references_returned
0,q05,vector,[RET-SupportingEvidence-3.1],[],True
1,q05,hybrid,[RET-SupportingEvidence-3.1],[],True
2,q05,hybrid_rerank,[RET-SupportingEvidence-3.1],[],True
3,q06,vector,[RET-ReturnProcedure-4.5],[],True
4,q06,hybrid,[RET-ReturnProcedure-4.5],[],True
5,q06,hybrid_rerank,[RET-ReturnProcedure-4.5],[],True
6,q17,vector,"[SHP-DeliveryProblems-7.3, SHP-DeliveryProblems-7.4]",[],True
7,q17,hybrid,"[SHP-DeliveryProblems-7.3, SHP-DeliveryProblems-7.4]",[],True
8,q17,hybrid_rerank,"[SHP-DeliveryProblems-7.3, SHP-DeliveryProblems-7.4]",[],True
9,q25,vector,"[REF-Refunds-6.1, REF-Refunds-6.2, REF-Refunds-6.3]",[],True


In [8]:
# Save per-question results and the three-row summary as separate JSON files.
output = ROOT / "results/rag"
output.mkdir(parents=True, exist_ok=True)
(output / "retrieval_results.json").write_text(json.dumps({
    "k": settings.k, "candidate_pool": settings.n_prime, "rrf_k": settings.rrf_k,
    "corpus_hash": manifest["corpus_hash"], "rows": rows,
}, indent=2) + "\n")
(output / "retrieval_summary.json").write_text(
    json.dumps(summary.reset_index().to_dict(orient="records"), indent=2) + "\n"
)

393

In [ ]:
# Inspect the expected, retrieved and referenced chunk IDs for one question.
CASE_ID = "q05"
case = next(case for case in cases if case["id"] == CASE_ID)
print(case["question"])
with pd.option_context("display.max_colwidth", None):
    display(results.loc[results.case_id == CASE_ID, [
        "approach", "expected_chunk_ids", "retrieved_chunk_ids", "reference_chunk_ids",
        "expected_reference_chunk_ids", "references_returned", "recall_at_k"
    ]].reset_index(drop=True))